# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [1]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [2]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [3]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [4]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

[1] Nom exact (mapping)     : 17067 | restants : 833
[2] Fuzzy nom (mapping)     :    55 | restants : 778


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [5]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

In [6]:
df_final

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,...,12.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,...,23.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,...,4.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17117,ITA-Serie A,2425,Parma,Mathias Løvik,NOR,"DF,MF",20,2003.0,6,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17118,ITA-Serie A,2425,Venezia,Mirko Marić,CRO,FW,29,1995.0,10,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17119,ITA-Serie A,2526,Genoa,Albert Grønbaek,DEN,MF,24-346,2001.0,4,1,...,16.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
17120,ITA-Serie A,2526,Parma,Mathias Løvik,NOR,MF,22-149,2003.0,9,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## **Les joueurs orphelins**

In [7]:
still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [8]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 685
Taux joueurs orphelins : 11.06%


In [9]:
orphelins_par_saison = (
    still_missing
    .groupby('season')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season
2021     12
2122     36
2223     48
2324     81
2425     49
2526    552
dtype: int64


In [10]:
top_orphelins = (
    still_missing['player']
    .value_counts()
    .head(20)
)

print(top_orphelins)

player
Babis Lykogiannis    6
Hianga'a Mbock       5
Jonathan Rowe        4
Álex Jiménez         4
Yellu Santiago       4
Karl Etta Eyong      4
Ben Gannon-Doak      3
Kosta Nedeljković    3
Manuel Ugarte        3
Benjamin Šeško       3
Marc Pubill          3
Abde Rebbach         3
Bertuğ Yıldırım      3
Abdulai Juma Bah     3
Ismaël Boura         3
Hákon Haraldsson     3
Merveille Papela     3
Keke Topp            3
Dženan Pejčinović    3
Domen Črnigoj        3
Name: count, dtype: int64
